In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

In [2]:
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

/Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c106detail19maybe_wrap_dim_slowIxEET_S2_S2_b
  Referenced from: <870081F6-12FD-3CEA-BC5C-30F4764F2A98> /Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torchvision/image.so
  Expected in:     <F2FE5CF8-5B5B-3FAD-ADF8-C77D90F49FC9> /Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torch/lib/libc10.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [3]:
import os

from torch.utils.data import Dataset, DataLoader

from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# 1. Chuẩn bị dữ liệu

- Bước 1: Đọc ảnh từ ổ cứng
- Bước 2: Biến đổi ảnh
- Bước 3: Tạo label tương ứng với ảnh

In [4]:
class MyBrainTumorDataset(Dataset):
    def __init__(self, data_folder, csv_path):
        self.data_folder = data_folder
        self.image_names = [name for name in os.listdir(data_folder) if name.endswith('.jpg')]

        # Transformation => Bước 2
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        
        # Label => Bước 3
        self.label_df = pd.read_csv(csv_path, usecols=['image_name', 'label'])
        self.image_names = [name for name in self.image_names if name in self.label_df.image_name.to_list()]
        
    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        # Buoc 1: Doc anh tu o cung
        image = Image.open(os.path.join(self.data_folder, image_name))

        # Buoc 2: Bien doi anh
        transformed_image = self.transform(image)

        # Buoc 3: Tao label cho anh
        label_str = self.label_df[self.label_df.image_name == image_name]['label'].values[0]
        if label_str == 'tumor':
            label = 1
        else:
            label = 0

        return transformed_image, label

In [5]:
train_dataset = MyBrainTumorDataset(
    data_folder='brain_tumor_dataset/train',
    csv_path='brain_tumor_dataset/brain_multi.csv'
)

In [6]:
len(train_dataset)

5744

In [7]:
train_dataset[0]

(tensor([[[-0.7333, -0.2784, -0.6706,  ..., -0.7961,  0.0588, -0.5843],
          [-0.7255, -0.4118, -0.6627,  ..., -0.8275, -0.0902, -0.7020],
          [-0.9765, -0.9529, -0.9843,  ..., -0.8353, -0.1608, -0.5373],
          ...,
          [-0.9922, -0.9922, -0.9922,  ..., -0.9922, -0.9922, -0.9922],
          [-0.9922, -0.9922, -0.9922,  ..., -0.9922, -0.9922, -0.9922],
          [-0.9922, -0.9922, -0.9922,  ..., -0.9922, -0.9922, -0.9922]],
 
         [[-0.7333, -0.2784, -0.6706,  ..., -0.7961,  0.0588, -0.5843],
          [-0.7255, -0.4118, -0.6627,  ..., -0.8275, -0.0902, -0.7020],
          [-0.9765, -0.9529, -0.9843,  ..., -0.8353, -0.1608, -0.5373],
          ...,
          [-0.9922, -0.9922, -0.9922,  ..., -0.9922, -0.9922, -0.9922],
          [-0.9922, -0.9922, -0.9922,  ..., -0.9922, -0.9922, -0.9922],
          [-0.9922, -0.9922, -0.9922,  ..., -0.9922, -0.9922, -0.9922]],
 
         [[-0.7333, -0.2784, -0.6706,  ..., -0.7961,  0.0588, -0.5843],
          [-0.7255, -0.4118,

In [8]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0
)
train_dataloader

# 2. Xây dựng mô hình

In [9]:
class MyCustomCNN(nn.Module):
    def __init__(self):
        super(MyCustomCNN, self).__init__()

        self.conv_1 = nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.relu_1 = nn.ReLU()
        self.pool_1 = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.conv_2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.relu_2 = nn.ReLU()
        self.pool_2 = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.linear_1 = nn.Linear(32 * 16 * 16, 128)
        self.relu_3 = nn.ReLU()

        self.linear_2 = nn.Linear(128, 2)
        self.softmax = nn.Softmax()

    def forward(self, x):
        x = self.conv_1(x)
        x = self.relu_1(x)
        x = self.pool_1(x)

        x = self.conv_2(x)
        x = self.relu_2(x)
        x = self.pool_2(x)
        
        x = x.view(-1, 32 * 16 * 16)
        x = self.linear_1(x)
        x = self.relu_3(x)
        
        x = self.linear_2(x)
        x = self.softmax(x)
        return x

In [10]:
model = MyCustomCNN()

In [11]:
model

MyCustomCNN(
  (conv_1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu_1): ReLU()
  (pool_1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu_2): ReLU()
  (pool_2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (linear_1): Linear(in_features=8192, out_features=128, bias=True)
  (relu_3): ReLU()
  (linear_2): Linear(in_features=128, out_features=2, bias=True)
  (softmax): Softmax(dim=None)
)

In [12]:
class MyResNetCNN(nn.Module):
    def __init__(self):
        super(MyResNetCNN, self).__init__()
        
        self.backbone = models.resnet18(pretrained=True)
        num_features = self.backbone.fc.in_features
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-1])
        
        self.linear = nn.Linear(num_features, 2)
        self.softmax = nn.Softmax()

    def forward(self, x):
        x = self.backbone(x)
        x = x.squeeze()
        x = self.linear(x)
        x = self.softmax(x)
        return x

In [13]:
model = MyResNetCNN()
model

/Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


MyResNetCNN(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

# 3. Huấn luyện mô hình

## 3.1. Khởi tạo hàm Loss và thuật toán tối ưu Optimizer

In [14]:
loss_func = nn.CrossEntropyLoss()
loss_func

CrossEntropyLoss()

In [15]:
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    lr: 0.001
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)

## 3.2. Huấn luyện và đánh giá mô hình

In [16]:
writer = SummaryWriter(log_dir='train_logs_1')
writer

2023-08-29 11:02:48.241181: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [17]:
num_epoch = 10

In [18]:
for epoch in range(num_epoch):
    # Train model
    model.train()
    for iteration_, (image, label) in enumerate(tqdm(train_dataloader, total=len(train_dataloader))):
        
        # Bước 1: Optimizer zero grad
        optimizer.zero_grad()

        # Bước 2: Foward data to model
        pred = model(image)

        # Bước 3: Tính giá trị loss
        loss_value = loss_func(pred, label)

        # Bước 4: Cập nhật trọng số của mô hình
        loss_value.backward()
        optimizer.step()
        
        global_iteration = epoch * len(train_dataloader) + iteration_
        writer.add_scalar('train_loss_iter', loss_value, global_iteration)

    # Bước 5: (Tuỳ chọn) In các thông số ra ngoài màn hình
    print(f'Epoch={epoch}', f'Training loss={loss_value.item()}')
    writer.add_scalar('train_loss_epoch', loss_value, epoch)

    # Evaluate model
    model.eval()
    with torch.no_grad():
        loss_sum = 0
        pred_list, label_list = [], []
        for image, label in tqdm(train_dataloader, total=len(train_dataloader)):
            pred = model(image)
            loss = loss_func(pred, label)
            loss_sum += loss.item()

            pred_list.append(pred)
            label_list.append(label)

        print(f'Test loss {loss_sum / len(train_dataloader)}')
        writer.add_scalar('test_loss_epoch', loss_value, epoch)
        
        # Calculate metrics
        final_pred = torch.concat(pred_list)
        final_pred = torch.argmax(final_pred, axis=1)
        final_label = torch.concat(label_list)
        
        epoch_accuracy_score = accuracy_score(final_pred, final_label)
        writer.add_scalar('test_accuracy_score_epoch', epoch_accuracy_score, epoch)

        epoch_precision_score = precision_score(final_pred, final_label)
        writer.add_scalar('test_precision_score_epoch', epoch_precision_score, epoch)

        epoch_recall_score = recall_score(final_pred, final_label)
        writer.add_scalar('test_recall_score_epoch', epoch_recall_score, epoch)

        epoch_f1_score = f1_score(final_pred, final_label)
        writer.add_scalar('test_f1_score_epoch', epoch_f1_score, epoch)

        print(classification_report(final_pred, final_label))
        
        torch.save(model.state_dict(), os.path.join('ckpt', f'ckpt_{epoch}.pth'))

  0%|                                                                                                                                                                                  | 0/2872 [00:00<?, ?it/s]/var/folders/kx/93nsnn7965sgpgl20_v6_28r0000gn/T/ipykernel_16225/1750391181.py:16: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = self.softmax(x)
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2872/2872 [06:58<00:00,  6.86it/s]


Epoch=0 Training loss=0.32633423805236816


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2872/2872 [02:15<00:00, 21.27it/s]


Test loss 0.47746641347625796
              precision    recall  f1-score   support

           0       0.13      0.91      0.23       173
           1       1.00      0.81      0.89      5571

    accuracy                           0.81      5744
   macro avg       0.56      0.86      0.56      5744
weighted avg       0.97      0.81      0.87      5744



  0%|                                                                                                                                                                                  | 0/2872 [00:00<?, ?it/s]/var/folders/kx/93nsnn7965sgpgl20_v6_28r0000gn/T/ipykernel_16225/1750391181.py:16: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = self.softmax(x)
  8%|████████████▊                                                                                                                                                           | 218/2872 [00:35<07:09,  6.18it/s]


KeyboardInterrupt: 

# 4. Predict new data

In [20]:
state_dict = torch.load('ckpt/ckpt_0.pth', map_location='cpu')
state_dict

OrderedDict([('backbone.0.weight',
              tensor([[[[-1.4520e-02, -7.3917e-03, -2.1949e-03,  ...,  6.0194e-02,
                          2.2602e-02, -1.0536e-02],
                        [ 6.8116e-03,  1.0985e-02, -1.0939e-01,  ..., -2.6812e-01,
                         -1.2646e-01,  7.7104e-03],
                        [-1.3339e-02,  5.9288e-02,  2.9665e-01,  ...,  5.2356e-01,
                          2.5964e-01,  6.5526e-02],
                        ...,
                        [-3.0633e-02,  1.8887e-02,  7.3958e-02,  ..., -3.3078e-01,
                         -4.1762e-01, -2.5637e-01],
                        [ 2.6079e-02,  4.0239e-02,  6.1062e-02,  ...,  4.1565e-01,
                          3.9268e-01,  1.6453e-01],
                        [-2.1084e-02, -7.8378e-03, -2.7862e-02,  ..., -1.5231e-01,
                         -8.5861e-02, -1.0034e-02]],
              
                       [[-1.5529e-02, -2.7864e-02, -3.5021e-02,  ...,  3.6088e-02,
                          6

In [21]:
new_model_1 = MyCustomCNN()

In [22]:
new_model_2 = MyResNetCNN()

/Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/minhnguyenhuu/miniconda3/envs/work_env/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [23]:
new_model_2.load_state_dict(state_dict)

<All keys matched successfully>

In [24]:
new_model_1.load_state_dict(state_dict)

RuntimeError: Error(s) in loading state_dict for MyCustomCNN:
	Missing key(s) in state_dict: "conv_1.weight", "conv_1.bias", "conv_2.weight", "conv_2.bias", "linear_1.weight", "linear_1.bias", "linear_2.weight", "linear_2.bias". 
	Unexpected key(s) in state_dict: "backbone.0.weight", "backbone.1.weight", "backbone.1.bias", "backbone.1.running_mean", "backbone.1.running_var", "backbone.1.num_batches_tracked", "backbone.4.0.conv1.weight", "backbone.4.0.bn1.weight", "backbone.4.0.bn1.bias", "backbone.4.0.bn1.running_mean", "backbone.4.0.bn1.running_var", "backbone.4.0.bn1.num_batches_tracked", "backbone.4.0.conv2.weight", "backbone.4.0.bn2.weight", "backbone.4.0.bn2.bias", "backbone.4.0.bn2.running_mean", "backbone.4.0.bn2.running_var", "backbone.4.0.bn2.num_batches_tracked", "backbone.4.1.conv1.weight", "backbone.4.1.bn1.weight", "backbone.4.1.bn1.bias", "backbone.4.1.bn1.running_mean", "backbone.4.1.bn1.running_var", "backbone.4.1.bn1.num_batches_tracked", "backbone.4.1.conv2.weight", "

In [30]:
new_model_2.eval()
with torch.no_grad():
    loss_sum = 0
    pred_list, label_list = [], []
    for image, label in tqdm(train_dataloader, total=len(train_dataloader)):
        pred = new_model_2(image)
        loss = loss_func(pred, label)
        loss_sum += loss.item()

        pred_list.append(pred)
        label_list.append(label)

    print(f'Test loss {loss_sum / len(train_dataloader)}')

    # Calculate metrics
    final_pred = torch.concat(pred_list)
    final_pred = torch.argmax(final_pred, axis=1)
    final_label = torch.concat(label_list)

    print(classification_report(final_pred, final_label))

  0%|                                                                                                                                                                                  | 0/2872 [00:00<?, ?it/s]/var/folders/kx/93nsnn7965sgpgl20_v6_28r0000gn/T/ipykernel_16225/1750391181.py:16: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = self.softmax(x)
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2872/2872 [02:39<00:00, 18.06it/s]

Test loss 0.4801389453491627
              precision    recall  f1-score   support

           0       0.12      0.94      0.21       155
           1       1.00      0.81      0.89      5589

    accuracy                           0.81      5744
   macro avg       0.56      0.88      0.55      5744
weighted avg       0.97      0.81      0.88      5744



In [ ]:
# Test loss 0.47746641347625796
#               precision    recall  f1-score   support

#            0       0.13      0.91      0.23       173
#            1       1.00      0.81      0.89      5571

#     accuracy                           0.81      5744
#    macro avg       0.56      0.86      0.56      5744
# weighted avg       0.97      0.81      0.87      5744